# Marketplace Safety - Prioritering av misstänkta annonser och meddelanden

**Bakgrund**  
Detta projekt är utfört på uppdrag av analysteamet på ett företag som driver en marknadsplats-app, liknande Blocket. Plattformen används dagligen av ett stort antal användare där de allra flesta är seriösa. Varje vecka förekommer dock en mindre andel problematiska aktiviteter: bluffannonser, spam, misstänkta konton som agerar snabbt samt försök att flytta konversationer utanför plattformen.

**Problem**  
Företagets Trust & Safety-team granskar och hanterar misstänkt innehåll manuellt. Problemet är att volymen är för stor för att teamet ska hinna granska allt i tid. Ledningen efterfrågar därför ett beslutsstöd som hjälper teamet att prioritera vad som ska granskas först.

**Mål**  
Målet är att leverera en lösning som:
- presterar rimligt bra på ny, osedd data
- går att köra regelbundet i produktion
- kan förklaras för icke-tekniska stakeholders

Fokus ligger på prioritering och beslutstöd.

**Stakeholder**   
Att fylla i på slutet av projektet (kravkortet är en hemlis  🤫)

Kolumnnamn | Förklaring |
|---|---|
| `id` | Unikt identifieringsnummer för varje post |
| `day` | Dagen då händelsen inträffade |
| `event_type` | Typ av händelse (t.ex. visning, meddelande, anmälan) |
| `category` | Produktkategori för annonsen |
| `region` | Geografiskt område där annonsen publicerades |
| `device` | Enhetstyp som användes (t.ex. mobil, dator) |
| `account_age_days` | Antal dagar sedan kontot skapades |
| `num_prev_listings` | Antal tidigare annonser från samma användare |
| `prev_reports_30d` | Antal gånger användaren anmälts de senaste 30 dagarna |
| `verification_level` | Verifieringsnivå för kontot |
| `price` | Annonsens pris |
| `num_images` | Antal bilder i annonsen |
| `message_length` | Längden på meddelandet i annonsen |
| `contains_off_platform` | Om användaren försökt flytta konversationen utanför plattformen |
| `urgency_words` | Om annonsen innehåller ord som skapar artificiell brådska |
| `payment_attempt` | Om ett betalningsförsök har skett |
| `time_to_first_response_min` | Tid i minuter till första svar |
| `is_suspicious` | Målvariabel — om annonsen är misstänkt (1) eller inte (0) |

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from joblib import dump, load

from sklearn.decomposition import PCA

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

sns.set_context("notebook")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8,4)

# 1) Data & EDA (Isabel)

- Visa datasetstorlek, datatyper och target-fördelning.
- Kontrollera saknade värden och beskriv hur ni hanterar dem.
- Minst 2 figurer/tabeller + kort tolkning.

In [ ]:
df = pd.read_csv("../data/historical_data.csv")
df.head()

In [ ]:
print("Datasets storlek:", df.shape)
print("---------------------------")

In [ ]:
print("Datatyper")
print("---------------------------")
df.info()

### Summering av dataset

- Datasetet innehåller 12,000 observationer och 18 variabler
- Text variabler är av typ *string* och numeriska variabler är av typerna *int64* och *float64*
- Det finns fyra kategoriska värden: *event_type*, *category*, *region*, *device*
- Target är *is_suspicious*: 0 = ej misstänkt, 1 = misstänkt

## Missing values

In [ ]:
print("-------------------------------")
print("Saknade värden per kolumn (antal):")
print("-------------------------------")
print(df.isna().sum())
print("-------------------------------")
print("Saknade värden per kolumn (%):")
print("-------------------------------")
df.isnull().sum() / len(df) * 100

### Hantering av saknade värden

Andelen saknade värden är relativt låg:
- *region*: 340 (2.83 %)
- *price*: 818 (6.81 %)
- *time_to_first_response_min*: 590 (4.91%)

Eftersom klassen är obalanserad är det mindre lämpligt att droppa rader, då procent av saknade värden inte är tillräckligt låg, i ett redan obalancerat problem kan minoiritetsklassen påverkas oproportioneligt. I detta fall är det istället bättre att imputera värdena.

Saknade värden kommer därför att imputeras i pipeline med hjälp av `SimpleImputer` för att minska data-läckage genom att imputering beräknas bara på träningsdatan och sedan att samma transformation appliceras på testdata. Det gör processen reproducerbar och säker i cross-validation. 

**Numeriska variabler:**
- Saknade värden kommer att ersättas med medianen (som är tålig mot outliers) för respektive numerisk kolumn. 
- `add_indicator=True` kommer att användas för att lägga till en extra binär kolumn för varje variabel som haft saknade värden: *0 = värdet fanns, 1 = värdet saknades*. För att modellen ska kunna lära sig om ett saknat värde tenderar att tillhöra en viss klass. 
- Efter imputering standardiseras variablerna med `StandardScaler`.

**Kategoriska variabler:**
- Sakande kategoriska värden kommer att ersättas med "unknown". 
- Därefter kommer `OneHotEncoder`att användas för att göra "unknown" till sin egen kategori, för att undvika felaktig information genom att fylla med en befintlig kategori. 
- `handle_unknow="ignore"` hjälper till med att hantera att nya kategorier i testdata inte orsakar fel.

## Definera X och y

In [ ]:
X_full = df.drop(["is_suspicious"], axis=1)
y_full = df["is_suspicious"]

print("X:", X_full.shape, "y:", y_full.shape)

## Klassfördelning av target (misstänkt aktivitet)

In [ ]:
class_dist = y_full.value_counts(normalize=True) * 100

ax = sns.barplot(
    x=class_dist.index,
    y=class_dist.values
)

plt.ylabel("Procent")
plt.xlabel("Misstänkt aktivitet (0 = Nej, 1 = Ja)")
plt.title("Klassfördelning av target (%)")

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%')

plt.show()

### Summering av klassfördelning
Klassfördelningen visar en tydlig obalans i datan: endast 10 % av de historiska observationerna är klassade som misstänkt aktivitet. Detta är dock förväntat, eftersom andelen misstänkt aktivitet normalt sett är låg.

## Korrelationer med target (misstänkt aktivitet)

Korrelationer visar hur starkt numeriska features separerar misstänkt aktivitet från icke-misstänkt. Kategoriska variabler utesluts eftersom korrelation mäter det linjära sambandet mellan två numeriska variabler.

In [ ]:
corr_with_target = X_full.copy()
corr_with_target["is_suspicious"] = y_full

corr_with_target = (
    corr_with_target
    .drop(["event_type", "category", "region", "device"], axis=1)
    .corr()["is_suspicious"]
    .sort_values(ascending=False)
)

print(corr_with_target)

### Summering korrelerande features

De två features som visar högst positiv korrelation med misstänkt aktivitet är *contains_off_platform* (0,14) och *prev_reports_30d* (0,13). Även om dessa samband är relativt svaga kan de ändå vara värdefulla i obalanserade klassificeringsproblem, eftersom effekten sannolikt är icke-linjär. Det finns troligen en tröskeleffekt, och dessa features kan ha större betydelse i kombination med andra variabler.

## Andel misstänka aktiviteter per kategori

Andelen misstänkt aktivitet analyseras för respektive kategorisk variabel för att undersöka om någon kategori sticker ut med en högre förekomst av misstänkt aktivitet.

In [ ]:
categorical_cols = [
    "event_type",
    "category",
    "region",
    "device"
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))  
axes = axes.flatten() 

temp_df = pd.concat([X_full, y_full.rename("is_suspicious")], axis=1)

for i, col in enumerate(categorical_cols):
    susp_count = (
        temp_df
        .groupby(col)["is_suspicious"]
        .mean()
        .sort_values(ascending=False) * 100
    )
    
    ax = sns.barplot(
        x=susp_count.index,
        y=susp_count.values,
        ax=axes[i]
    )
    
    ax.set_ylabel("Procent")
    ax.set_title(f"Misstänkt aktivitet per {col}")
    
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%')

plt.tight_layout()
plt.show()

### Summering av misstänkt aktivitet per kategori
Analysen av historisk data visar inga tydliga skillnader mellan kategorier när det gäller andelen misstänkt aktivitet. Fördelningen av misstänkta händelser inom de olika kategorierna speglar i stort den övergripande klassfördelningen och ligger konsekvent runt 10 %.

# 2) Pipeline & preprocessing (Irene)

- Skapa en train/test-split från historical_data.csv.
- Bygg en pipeline där preprocessing sker på ett sätt som undviker att testdata påverkar träningen (undvik leakage).
- För klassificering: använd gärna stratified split så klasserna fördelas rimligt.

In [ ]:
df.describe().T

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    random_state=42,
    stratify=y_full
)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)
print("-------------------------------")
print(f"Classification train: \n{y_train.value_counts(normalize=True)}")
print("-------------------------------")
print(f"Classification test: \n{y_test.value_counts(normalize=True)}")
print("-------------------------------")

### Summering av train/test split

Vi delar upp datan i tränings- och testset med en 80/20-fördelning. **stratify=y_full** garanterar att andelen misstänkta och icke-misstänkta fall är proportionerligt lika i både tränings- och testset.

In [ ]:
def engineer_features(X):
    X = X.copy()
    X["report_rate"] = X["prev_reports_30d"] / (X["num_prev_listings"] + 1)
    X["risk_score"] = X["contains_off_platform"] + X["urgency_words"] + X["payment_attemp"]
    return X

feature_engineer = FunctionTransformer(engineer_features)

### Summering av feature engineering

Vi skapar två nya features baserade på befintliga kolumner:
- **report_rate** normaliserar antalet anmälningar mot användarens aktivitetsnivå
- **risk_score** summerar tre binära varningssignaler till ett enda riskindex

Feature engineering handlar om att skapa nya variabler från den data vi redan har, med målet att göra mönstren tydligare för modellen.  

Dessa transformationer är inlindade i en FunctionTransformer så att de körs inuti pipelinen och aldrig appliceras före train/test-splitten.

In [ ]:
numeric_features = ["account_age_days", "num_prev_listings", "prev_reports_30d",
    "price", "num_images", "message_length", "verification_level",
    "time_to_first_response_min", "payment_attempt",
    "contains_off_platform", "urgency_words",
    # engineered features:
    "report_rate", "risk_score"]

categorical_features = ["event_type", "category", "region", "device"]

### Summering av feature selection

De numeriska och kategoriska variablerna separerades eftersom de kräver olika typer av förbehandling:  
- Numeriska variabler behöver skalas med StandardScaler för att en model son Logistic Regression inte ska påverkas av att olika kolumner har olika storleksordning.  
- Kategoriska variabler behöver istället omvandlas till binära kolumner med OneHotEncoder.

Kolumnerna **id** och **day** exkluderades eftersom det första är ett unikt identifieringsnummer utan prediktivt värde, medans den andra är ett gränsfallsbeslut som kan leda till att modellen lär sig slumpen istället förverkliga beteendemönster.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

### Summering av transformers

Vi definierar två separata transformers:
- Den numeriska transformern hanterar saknade värden med medianvärdet och använder add_indicator=True för att skapa en binär kolumn som markerar var värden saknades. Detta ger modellen möjlighet att lära sig om det finns ett samband mellan saknade värden och misstänkt aktivitet. 
- Därefter standardiseras värdena med StandardScaler. Den kategoriska transformern fyller saknade värden med konstanten "Unknown" så att missingness bevaras som en egen kategori, följt av OneHotEncoder som omvandlar textvärden till binära kolumner som modellen kan tolka.

Anledningen är att ett saknat värde i dessa kolumner potentiellt är informativt i sig: en säljare som döljer priset eller aldrig svarar kan vara ett tecken på misstänkt aktivitet.

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

Vi kombinerar de två transformers i en ColumnTransformer som applicerar rätt transformer på rätt kolumner.

**remainder="drop"** säkerställer att kolumner som inte finns i någon av listorna ignoreras.

In [ ]:
full_pipeline = Pipeline(steps=[
    ("features", feature_engineer),
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(random_state=42, max_iter=1000))
])

full_pipeline

### Summering av pipeline

Vi sätter ihop alla steg i en sammanhängande Pipeline:   
- feature engineering  
- preprocessing  
- modell  

Fördelen med en pipeline är att alla transformationer appliceras automatiskt och konsekvent på både tränings- och testdata, vilket eliminerar risken för data leakage.  

För att byta model kan vi bara byta ut LogisticRegression mot en annan och allt annat fungerar automatiskt.

# 3) Modelljämförelse (Nora)

- Skapa en baseline.
- Träna minst två ytterligare modeller (minst 3 totalt inkl baseline).
- Utvärdera med rimlig metod (t.ex. cross-validation på train eller tydligt valideringsupplägg)
- Välj metric och motivera valet utifrån ert kravkort.
- (Exempel på modeller: LogisticRegression, DecisionTree, RandomForest, GradientBoosting…)

# 4) Optimering (Ummulbanin)

- Välj en “final” modell baserat på jämförelsen.
- Gör tuning på den valda modellen (litet grid, minst 1–2 parametrar).
- Förklara kort vad ni optimerade och varför (koppla till kravkortet).

# 5) Threshold/prioritering (Abdullahi)

Ni måste bestämma hur modellen ska användas i praktiken. Välj en strategi:

    A) Threshold-beslut
    - flagga misstänkt om proba ≥ t
    - motivera t utifrån kravkortet och visa konsekvenser (FP/FN eller precision/recall)
    
    B) Top-X prioritering
    - flagga de X% högst risk (t.ex. top 5% eller top 50 per dag)
    - motivera X utifrån kravkortet och visa konsekvenser
    - Ni ska tydligt visa hur ert val påverkar FP/FN och varför det passar er stakeholder.

# 6) Deploy-test: ny data (tisdag kursvecka 6)

- När ni får new_data.csv ska ni:
- använda er låsta pipeline
- skapa prediktioner och en prioriteringslista